In [65]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [67]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [68]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'CumulativeAssetCoveredLengthKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'PeakAboveSATCount',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'Bm1Density',
        'Bm2Density',
        'B0Share',
        'B1Share',
        'Bm1Share',
        'Bm2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare',
        'POR',
        'CurrentCompletion'
    ]


In [69]:
query = f"""DROP VIEW IF EXISTS Weekly_KPI;"""
cursor.execute(query)
conn.commit()


In [70]:
query = """
CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Week'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [71]:
print(query)


CREATE VIEW IF NOT EXISTS Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WH

In [72]:
query = "SELECT * FROM Weekly_KPI WHERE Year = 2026 AND BoundaryRegion IS NULL;"
df = pd.read_sql_query(query, conn)
conn.close()

In [73]:
Query(query = f"SELECT * FROM KPI_Data WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = 'Cadent') AND KPIId = 'POR'").execute(KPIHub_Conn)

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,POR_Cadent_Y2023_W14,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,14,None,None,2026-07-08 20:50:16.003653
1,POR_Cadent_Y2023_W15,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,15,None,None,2026-07-08 20:50:16.003653
2,POR_Cadent_Y2023_W16,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,16,None,None,2026-07-08 20:50:16.003653
3,POR_Cadent_Y2023_W17,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,17,None,None,2026-07-08 20:50:16.003653
4,POR_Cadent_Y2023_W19,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Week,19,None,None,2026-07-08 20:50:16.003653
...,...,...,...,...,...,...,...,...,...,...
164,POR_Cadent_Y2026_W24,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,24,66.0,None,2026-07-08 20:50:16.003653
165,POR_Cadent_Y2026_W25,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,25,68.0,None,2026-07-08 20:50:16.003653
166,POR_Cadent_Y2026_W26,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,26,70.0,None,2026-07-08 20:50:16.003653
167,POR_Cadent_Y2026_W27,POR,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2026,Week,27,73.0,None,2026-07-08 20:50:16.003653


In [74]:
df

,Year,PeriodValue,CustomerName,BoundaryRegion,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,CumulativeAssetCoveredLengthKm,ServicePipeKm,...,Bm2Density,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare,POR,CurrentCompletion
0,2026,1,Cadent,None,50.53,47.68,49.69,47.03,47.68,0.84,...,0.06,23.08,0.00,73.63,3.30,73.47,19.39,0.07,127000.0,0.0
1,2026,2,Cadent,None,229.57,216.26,226.35,213.85,263.94,3.22,...,0.06,28.70,0.87,65.22,5.22,67.58,22.27,0.10,127000.0,0.0
2,2026,3,Cadent,None,312.37,282.81,302.16,276.15,546.76,10.21,...,0.04,38.41,2.65,54.97,3.97,73.48,18.60,0.08,127000.0,0.0
3,2026,4,Cadent,None,339.23,321.31,332.75,315.89,868.07,6.48,...,0.07,24.06,0.58,69.28,6.09,71.77,20.97,0.07,127000.0,1.0
4,2026,5,Cadent,None,716.21,671.67,699.31,657.47,1539.73,16.90,...,0.07,22.18,0.75,68.80,8.27,63.11,26.90,0.10,127000.0,1.0
5,2026,6,Cadent,None,1530.09,1412.45,1491.98,1383.53,2952.19,38.11,...,0.09,26.22,1.03,64.65,8.10,68.72,21.59,0.10,127000.0,2.0
6,2026,7,Cadent,None,1442.95,1327.13,1408.33,1303.17,4279.32,34.62,...,0.10,21.18,0.57,68.63,9.62,70.39,20.06,0.10,127000.0,3.0
7,2026,8,Cadent,None,1647.92,1560.83,1618.80,1539.95,5840.15,29.12,...,0.07,25.51,0.51,65.68,8.31,73.13,20.76,0.06,127000.0,5.0
8,2026,9,Cadent,None,1466.42,1393.40,1435.40,1369.36,7233.55,31.02,...,0.09,24.17,0.93,65.56,9.34,75.14,16.83,0.08,127000.0,6.0
9,2026,10,Cadent,None,1210.19,1134.14,1191.31,1119.10,8367.69,18.87,...,0.15,16.87,2.08,68.26,12.79,71.97,14.85,0.13,127000.0,7.0
